# Audio RAG / Music recommendations — local learning notebook
## Complete local lab: music recommendations by example audio
Run each code cell using **Shift+Enter**. The complete notebook now covers preprocessing, PANNs embeddings, Chroma indexing, recommendation, playback and evaluation.

**Source:** [Ironhack lab](https://github.com/ironhack-labs/lab-music-recommendations), starter notebook reviewed on 2 November 2026. An unchanged copy is preserved in `lab_original/`.

The starter demonstrates similarity search on **ESC-50 environmental sounds** (not a music catalogue). It uses **PANNs**, 2048-dimensional embeddings, Pinecone and playback of nearest matches. The assignment asks you to extend the idea to recommending songs using music datasets and user preferences.

**Our adaptation:** local Windows CPU + Jupyter + persistent ChromaDB. No Colab or Pinecone cloud is needed for this version. No personal smartphone recordings are used.

### Concepts carried over from the image project
Audio file → waveform → consistent sample rate/channel handling → audio encoder → embedding → ChromaDB cosine search → deduplicated ranked tracks → audio playback.

CLIP is replaced by an audio encoder. PANNs is the starter model; we will check its input sample rate and CPU requirements before implementing it. For song recommendations we must decide what the user supplies: an example song, musical preferences as metadata, or text descriptions. PANNs audio embeddings alone do not provide a text-query encoder. A music-oriented or audio-text model may be appropriate for the extension, depending on the chosen query mode.

This project performs retrieval and produces grounded, template-based explanations. It does not use a generative LLM or pretend to support arbitrary text-to-audio similarity.

### Cell 1 — project and environment
Only standard-library checks. This does not install packages or load any model.

In [ ]:
from pathlib import Path
import sys, shutil, importlib.util, json
PROJECT = Path.cwd()
AUDIO_PATH = PROJECT / "data" / "miaow_16k.wav"
assert AUDIO_PATH.is_file(), f"Sample missing: {AUDIO_PATH}"
print("Python:", sys.version.split()[0])
print("Project:", PROJECT)
print("Free C: space (GiB):", round(shutil.disk_usage(PROJECT).free / 1024**3, 2))
for name in ("torch", "numpy", "librosa", "soundfile", "panns_inference"):
    print(f"{name}: {'available' if importlib.util.find_spec(name) else 'not installed'}")
chroma_path = PROJECT.parent / "Local_Image_RAG" / "vendor_py" / "chromadb"
print("Shared local Chroma files:", chroma_path.is_dir())
print("No model loaded and no index created in Step 1.")
print("Audio dependencies are stored locally in:", PROJECT / "vendor_py")

### Cell 2 — inspect the original WAV file
**Sample rate** is samples per second. **Channels** tells us mono/stereo. Duration = frames ÷ sample rate. The sample's filename is not a substitute for checking its actual header.

This file is the cat-sound query supplied in the lab. It is a small plumbing example, not our final music recommendation dataset. We will resample actual waveform data when required by the encoder; merely changing the playback rate would alter pitch/speed.

In [ ]:
import wave, hashlib
with wave.open(str(AUDIO_PATH), "rb") as wav:
    info = {
        "filename": AUDIO_PATH.name,
        "sample_rate_hz": wav.getframerate(),
        "channels": wav.getnchannels(),
        "sample_width_bytes": wav.getsampwidth(),
        "frames": wav.getnframes(),
        "duration_seconds": round(wav.getnframes() / wav.getframerate(), 3),
        "compression": wav.getcomptype(),
        "file_bytes": AUDIO_PATH.stat().st_size,
        "sha256": hashlib.sha256(AUDIO_PATH.read_bytes()).hexdigest(),
    }
assert info["sample_rate_hz"] > 0 and info["frames"] > 0
assert info["compression"] == "NONE", "Inspect non-PCM audio before continuing"
print(json.dumps(info, indent=2))
(PROJECT / "step1_audio_inspection.json").write_text(json.dumps(info, indent=2))

### Cell 3 — listen
Press the player’s Play button. Audio is read from the local file; it does not autoplay.

In [ ]:
from IPython.display import Audio, display
# embed=False keeps the notebook small; Jupyter serves this local audio file.
display(Audio(url="data/miaow_16k.wav", autoplay=False))

## Step 1 complete
The environment, original WAV properties and player above are the first checkpoint. Continue below only for Step 2.

# Step 2 — prepare the waveform for the lab’s PANNs encoder
**Goal:** convert the original PCM WAV to mono float32 audio and resample it to **32,000 Hz**, as used by the [official PANNs inference example](https://github.com/qiuqiangkong/panns_inference).

We retain the original file. Upsampling 16 kHz audio to 32 kHz does not add missing acoustic detail; it satisfies the model input format while preserving duration and playback speed. No model, embeddings or Chroma audio index are created in this step.

### Cell 4 — load and resample
The helper explicitly accepts uncompressed 16-bit PCM WAV files. Other formats will be handled later, rather than silently decoded incorrectly. Averaging channels produces mono; very different stereo channels may partially cancel, so inspect source recordings before applying this broadly.

In [ ]:
import math
import numpy as np
from scipy.signal import resample_poly

TARGET_SR = 32_000

def prepare_pcm16_wav(path, target_sr=TARGET_SR):
    with wave.open(str(path), "rb") as wav:
        if wav.getcomptype() != "NONE" or wav.getsampwidth() != 2:
            raise ValueError("This helper requires an uncompressed 16-bit PCM WAV.")
        source_sr = wav.getframerate()
        channels = wav.getnchannels()
        frames = wav.getnframes()
        raw = wav.readframes(frames)
    if frames == 0 or source_sr <= 0:
        raise ValueError("Empty or invalid audio")
    waveform = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0
    waveform = waveform.reshape(-1, channels).mean(axis=1)
    assert np.isfinite(waveform).all()
    if source_sr != target_sr:
        divisor = math.gcd(source_sr, target_sr)
        waveform = resample_poly(waveform, target_sr // divisor,
                                 source_sr // divisor).astype(np.float32)
    # Filtering can create tiny overshoots; apply a documented gain only if needed.
    peak = float(np.max(np.abs(waveform)))
    gain = 1.0 / peak if peak > 1.0 else 1.0
    waveform = np.ascontiguousarray(waveform * gain, dtype=np.float32)
    metadata = {"source_sample_rate_hz": source_sr, "source_channels": channels,
                "source_frames": frames, "target_sample_rate_hz": target_sr,
                "output_samples": int(waveform.size), "peak_gain_applied": gain}
    return waveform, metadata

audio_32k, preprocessing = prepare_pcm16_wav(AUDIO_PATH)
batch = audio_32k[None, :]  # model input: (batch_size, samples)
original_duration = preprocessing["source_frames"] / preprocessing["source_sample_rate_hz"]
prepared_duration = len(audio_32k) / TARGET_SR
assert audio_32k.dtype == np.float32 and batch.ndim == 2
assert np.isfinite(batch).all() and np.max(np.abs(batch)) <= 1.00001
assert abs(original_duration - prepared_duration) <= 1 / TARGET_SR
print("Original duration:", round(original_duration, 4), "seconds")
print("Prepared duration:", round(prepared_duration, 4), "seconds")
print("Prepared rate:", TARGET_SR, "Hz | dtype:", audio_32k.dtype)
print("Model-ready shape:", batch.shape)
print(json.dumps(preprocessing, indent=2))

### Cell 5 — inspect and listen
The plot shows amplitude over time. Play the prepared version and compare it with Step 1; the pitch and speed should stay the same.

In [ ]:
import matplotlib.pyplot as plt
seconds = np.arange(len(audio_32k)) / TARGET_SR
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.plot(seconds, audio_32k, linewidth=0.5)
ax.set(xlabel="Time (seconds)", ylabel="Amplitude", title="Prepared mono waveform — 32 kHz")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()
display(Audio(audio_32k, rate=TARGET_SR, normalize=False, autoplay=False))

### Cell 6 — save the prepared array and provenance
We save the original file hash and preprocessing settings beside the NumPy array so later embeddings can be traced to the source.

In [ ]:
prepared_dir = PROJECT / "prepared"
prepared_dir.mkdir(exist_ok=True)
np.save(prepared_dir / "miaow_32k_mono.npy", audio_32k, allow_pickle=False)
preprocessing.update({"source_path": str(AUDIO_PATH),
                      "source_sha256": hashlib.sha256(AUDIO_PATH.read_bytes()).hexdigest(),
                      "duration_seconds": prepared_duration,
                      "output_dtype": str(audio_32k.dtype)})
(prepared_dir / "miaow_preprocessing.json").write_text(json.dumps(preprocessing, indent=2))
restored = np.load(prepared_dir / "miaow_32k_mono.npy", allow_pickle=False)
assert np.array_equal(restored, audio_32k)
print("Step 2 verified: prepared array saved and read back successfully.")
print("No embedding model has been loaded.")

## Step 2 complete
The original audio is preserved. Continue below to load the encoder and music catalogue.

# Step 3 — load PANNs Cnn14 on CPU
The official lab uses Cnn14 trained on AudioSet. It returns **2048-dimensional embeddings**. We use the same pretrained architecture, 32 kHz mono audio and cosine similarity. This is a general acoustic model, not a model trained on this tiny music collection.

The helper loads the already downloaded official checkpoint locally with `weights_only=True`; it does not call wget or download anything during notebook execution. It sets CPU threads to at most four and processes one 10-second window at a time. Up to three evenly spaced windows are averaged into one track vector and L2-normalized.

The supplied `audio_rag.py` contains readable helpers; important methods are also shown below. Checkpoint source: [PANNs pretrained models](https://zenodo.org/records/3987831).

In [ ]:
import sys, importlib, time
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import audio_rag
importlib.reload(audio_rag)
from audio_rag import AudioEncoder, MusicRAG, load_audio, load_catalogue, explain, file_hash
started = time.perf_counter()
encoder = AudioEncoder()
print("PANNs Cnn14 ready on CPU in", round(time.perf_counter() - started, 1), "seconds")

In [ ]:
import inspect
from IPython.display import Code
print("How an audio track becomes one normalized vector:")
display(Code(inspect.getsource(AudioEncoder.embed), language="python"))
example_vector = np.asarray(encoder.embed(audio_32k))
assert example_vector.shape == (2048,)
assert np.isclose(np.linalg.norm(example_vector), 1.0, atol=1e-5)
print("Embedding shape:", example_vector.shape)
print("Vector length:", round(float(np.linalg.norm(example_vector)), 5))
print("First eight values:", example_vector[:8])

# Step 4 — a real, small music catalogue
We use **Tiny AAM**, the small sample linked by the authors of the lab's suggested Artificial Audio Multitracks dataset. It contains **20 algorithmically composed tracks**. These are musical recordings, not the ESC-50 environmental-sound demo or personal smartphone recordings.

**Attribution:** Fabian Ostermann and Igor Vatolkin, *Tiny AAM: Sample from the Artificial Audio Multitracks Dataset*, [Zenodo 6771120](https://zenodo.org/records/6771120), CC BY 4.0. The dataset archive and model checksums were verified during setup. Full mixes only are extracted; instrument names come from stem filenames, and BPM is calculated from the provided beat timestamps. We do not invent song titles, artist names, genres or moods.

The split is fixed: every fifth track in sorted order is held out (4); the other 16 form the index. All windows belonging to a track stay together. This is a small educational demonstration, not a catalogue of commercial songs.

In [ ]:
import pandas as pd
catalogue = load_catalogue()
reference_tracks = [r for r in catalogue if r["split"] == "reference"]
heldout_tracks = [r for r in catalogue if r["split"] == "heldout"]
assert len(reference_tracks) == 16 and len(heldout_tracks) == 4
assert not {r["track_id"] for r in reference_tracks} & {r["track_id"] for r in heldout_tracks}
display(pd.DataFrame(catalogue)[["track_id", "bpm", "instruments", "split"]])

# Step 5 — index the reference tracks in local ChromaDB
Only the 16 reference tracks are indexed. Each record stores the normalized audio embedding, source-file hash, track ID, path, BPM, instruments, licence and source. The collection uses **cosine distance**; similarity = 1 − distance.

Repeated indexing reuses unchanged embeddings by SHA-256 and updates metadata. Multiple windows are represented by one aggregate vector per track; recommendations also deduplicate track IDs. No delete-index cell is included. If changing model or preprocessing, use a new collection and regenerate vectors.

In [ ]:
rag = MusicRAG(encoder, collection_name="tiny_aam_cnn14_v1")
index_count = rag.index(reference_tracks)
print("Vectors in persistent local index:", index_count)
assert index_count == len(reference_tracks)
indexed_ids = {m["track_id"] for m in rag.collection.get()["metadatas"]}
assert not indexed_ids & {r["track_id"] for r in heldout_tracks}
print("Held-out tracks are absent from the index.")

# Step 6 — recommend similar music and explain the evidence
The query expresses the user's preference through an **example music recording**. Optional BPM and instrument constraints are applied to the retrieved catalogue metadata. These filters are explicit preferences, not inferred emotions.

The query's exact file hash is excluded if it appears in the index. Cosine similarity is a ranking score, **not a probability or a percentage of musical preference**. PANNs may favour timbre or instrumentation over melody. Explanations below are deterministic summaries of retrieved evidence, not invented LLM descriptions.

In [ ]:
def show_tracks(matches):
    print(explain(matches))
    for match in matches:
        print("Track", match["track_id"], "| score", round(match["similarity"], 3))
        relative = Path(match["path"]).relative_to(PROJECT).as_posix()
        display(Audio(url=relative, autoplay=False))

query_track = heldout_tracks[0]
print("UNSEEN QUERY:", query_track["track_id"], "| BPM", query_track["bpm"])
display(Audio(url=Path(query_track["path"]).relative_to(PROJECT).as_posix(), autoplay=False))
matches = rag.recommend(query_track["path"], k=3)
assert len(matches) == 3
assert all(m["sha256"] != file_hash(query_track["path"]) for m in matches)
show_tracks(matches)

In [ ]:
# Inspect the actual Chroma query logic used by the helper.
display(Code(inspect.getsource(MusicRAG.recommend), language="python"))

# Example explicit preference: retrieve music containing Piano, between 60 and 180 BPM.
filtered = rag.recommend(query_track["path"], k=3, min_bpm=60, max_bpm=180, instrument="Piano")
show_tracks(filtered)  # An empty result is valid: the filters may be too restrictive.

## Spotify song - listening and comparison
Paste a **single Spotify song link** from Share > Copy Song Link, then click **Show Spotify player**. This embeds the official Spotify player; playback availability depends on Spotify and your account. This player needs internet but no developer API key.

**This is playback only.** The notebook does not download, record, inspect or send Spotify audio/metadata into PANNs. A song link is not an audio waveform. Spotify prohibits ingesting its content into AI models: [Developer Policy](https://developer.spotify.com/policy).

For actual audio similarity results, use the file-upload control immediately below with a separately obtained WAV/MP3/FLAC/OGG file that you have permission to analyse. Spotify offline downloads are not ordinary MP3 files. Results still come from the 16-track Tiny AAM reference catalogue, not from Spotify's entire catalogue.

In [ ]:
import ipywidgets as widgets
from IPython.display import IFrame, display
from spotify_player import spotify_embed_url
spotify_link = widgets.Text(description="Song link:", placeholder="https://open.spotify.com/track/...", layout=widgets.Layout(width="90%"))
spotify_button = widgets.Button(description="Show Spotify player")
spotify_output = widgets.Output()

def show_spotify_player(_):
    with spotify_output:
        spotify_output.clear_output()
        try:
            url = spotify_embed_url(spotify_link.value)
            display(IFrame(src=url, width="100%", height=152))
            print("Playback only. To run similarity search, upload a separate permitted audio file below.")
        except ValueError as exc:
            print(str(exc))

spotify_button.on_click(show_spotify_player)
display(spotify_link, spotify_button, spotify_output)

## Step 7B - browser microphone recorder (recommended)
The older Windows/PortAudio input failed on this laptop. This version uses Chrome's microphone capture instead.

1. Run earlier cells through Step 6, then this cell.
2. Open the recorder link below (or use its embedded panel).
3. Play the phone speaker nearby and click **Record 20 seconds and recommend**.
4. If Chrome asks, choose **Allow microphone** yourself. Recording starts after permission is granted.
5. At the end of the countdown, **pause the phone**. The captured recording and recommended-track players will appear below the button. Press ▶ to listen.

No microphone is accessed until you click Record. Audio is sent only to the loopback recorder service on this laptop and saved in ignored `data/microphone_queries/`. The service uses the already-loaded PANNs model and local Chroma catalogue. It stops if the kernel is restarted; rerun this cell to start it again. Only use audio you may capture and analyse. This finds similar tracks in the small reference catalogue; it does not identify a Spotify song.

The recorder captures **20 seconds** and returns **up to 5 tracks**. The expanded FMA collection is separate from the original Tiny AAM teaching/evaluation collection. Reference tracks have titles, artists and dataset genre tags; retrieval uses acoustic embeddings, so a genre match is not guaranteed. Run `expand_catalogue.py` and `build_expanded_index.py` to rebuild this collection.


In [ ]:
from browser_microphone import start_browser_recorder
from IPython.display import IFrame, HTML, display

# The live recorder uses the expanded collection when active_catalogue.json exists.
recorder_rag = MusicRAG(encoder)
print("Recorder catalogue:", recorder_rag.collection.count(), "tracks")
browser_recorder_url = start_browser_recorder(recorder_rag, PROJECT)
display(HTML(f'<p><a href="{browser_recorder_url}" target="_blank">Open the browser microphone recorder in a separate tab</a></p>'))
display(IFrame(src=browser_recorder_url, width="100%", height=650,
               extras=['allow="microphone"']))
print("Ready. Click Record in the browser recorder when your phone is playing.")

# Step 7 — interactive recommendation
Choose one of the four unseen songs, or upload your own WAV/MP3/FLAC/OGG file. Uploaded audio is saved locally under `data/query_uploads/` only. It is not automatically added to the catalogue or sent to a remote inference API. Keep one song per upload; long recordings are represented by at most three 10-second windows.

Music is the supported domain. The cat sample from Step 1 is useful for pipeline checks, but music recommendations for it would not be meaningful: nearest-neighbour search always ranks available items and is not an out-of-domain detector.

In [ ]:
import ipywidgets as widgets
import re
choose_track = widgets.Dropdown(options=[(r["track_id"], r["path"]) for r in heldout_tracks], description="Query song:")
upload_audio = widgets.FileUpload(accept=".wav,.mp3,.flac,.ogg", multiple=False)
use_filters = widgets.Checkbox(value=False, description="Apply preferences")
min_tempo = widgets.IntText(value=60, description="Min BPM:")
max_tempo = widgets.IntText(value=180, description="Max BPM:")
instrument_filter = widgets.Text(value="", description="Instrument:", placeholder="e.g. Piano")
recommend_button = widgets.Button(description="Recommend music", button_style="primary")
results_box = widgets.Output()

def on_recommend(_):
    with results_box:
        results_box.clear_output()
        recommend_button.disabled = True
        try:
            path = Path(choose_track.value)
            if upload_audio.value:
                entry = next(iter(upload_audio.value.values())) if isinstance(upload_audio.value, dict) else upload_audio.value[0]
                extension = Path(entry["name"]).suffix.lower()
                if extension not in (".wav", ".mp3", ".flac", ".ogg"):
                    raise ValueError("Choose WAV, MP3, FLAC or OGG audio.")
                payload = bytes(entry["content"])
                if len(payload) > 25 * 1024**2:
                    raise ValueError("For this local demo, use a file smaller than 25 MB.")
                folder = PROJECT / "data" / "query_uploads"
                folder.mkdir(exist_ok=True)
                path = folder / (hashlib.sha256(payload).hexdigest()[:16] + extension)
                path.write_bytes(payload)
            print("Query:", path.name)
            display(Audio(url=path.relative_to(PROJECT).as_posix(), autoplay=False))
            options = {"min_bpm": min_tempo.value, "max_bpm": max_tempo.value,
                       "instrument": instrument_filter.value.strip() or None} if use_filters.value else {}
            show_tracks(rag.recommend(path, k=5, **options))
        except Exception as exc:
            print("Could not recommend:", str(exc))
        finally:
            recommend_button.disabled = False

recommend_button.on_click(on_recommend)
display(choose_track, upload_audio, use_filters, min_tempo, max_tempo,
        instrument_filter, recommend_button, results_box)

# Step 8 — evaluate on held-out songs
A successful API call is not evidence that recommendations are good. We report instrumentation overlap (Jaccard) and BPM difference for top-3 results versus a seeded random catalogue baseline. These are **weak metadata proxies**, not human preference accuracy, and PANNs was not trained to optimize them. With only four queries, do not generalize these measurements to other music.

Listen to the query and returned songs and record your own judgement below. No artist-disjoint performance claim is made: this sample consists of generated tracks and may share compositional patterns.

In [ ]:
def instrument_set(row):
    return {x.strip() for x in row["instruments"].split(",") if x.strip()}
def overlap(a, b):
    aa, bb = instrument_set(a), instrument_set(b)
    return len(aa & bb) / len(aa | bb) if aa | bb else 0.0

rng = np.random.default_rng(42)
evaluation = []
for query in heldout_tracks:
    retrieved = rag.recommend(query["path"], k=3)
    baseline = [reference_tracks[i] for i in rng.choice(len(reference_tracks), size=3, replace=False)]
    assert query["track_id"] not in {m["track_id"] for m in retrieved}
    assert len({m["track_id"] for m in retrieved}) == len(retrieved)
    evaluation.append({
        "query": query["track_id"],
        "retrieval_instrument_jaccard": float(np.mean([overlap(query,m) for m in retrieved])),
        "random_instrument_jaccard": float(np.mean([overlap(query,m) for m in baseline])),
        "retrieval_mean_bpm_difference": float(np.mean([abs(query["bpm"]-m["bpm"]) for m in retrieved])),
        "random_mean_bpm_difference": float(np.mean([abs(query["bpm"]-m["bpm"]) for m in baseline])),
        "retrieved_track_ids": ", ".join(m["track_id"] for m in retrieved),
    })
evaluation_df = pd.DataFrame(evaluation)
display(evaluation_df)
display(evaluation_df.select_dtypes(include="number").mean().to_frame("mean across 4 queries"))
evaluation_df.to_csv(PROJECT / "evaluation.csv", index=False)
print("No self-matches or duplicate result tracks in the four held-out queries.")

In [ ]:
# A catalogue query must exclude itself too (not just unseen test tracks).
self_test = rag.recommend(reference_tracks[0]["path"], k=3)
assert reference_tracks[0]["track_id"] not in {m["track_id"] for m in self_test}
# Reopening the persistent DB should retain the index without embedding again.
reopened = MusicRAG(encoder, collection_name="tiny_aam_cnn14_v1")
assert reopened.collection.count() == 16
assert rag.recommend(query_track["path"], min_bpm=9999) == []
verification = {"reference_tracks":16, "heldout_tracks":4, "embedding_dimensions":2048,
                "sample_rate_hz":32000, "self_exclusion_passed":True,
                "persistence_passed":True, "empty_filter_passed":True,
                "evaluation":evaluation}
(PROJECT / "verification.json").write_text(json.dumps(verification, indent=2))
print("Pipeline checks passed. See verification.json and evaluation.csv.")

### Your listening assessment
After playing results, edit this cell with your observations:

| Query track | Useful recommendations (0–3) | Similar instruments/tempo? | Other observations |
|---|---:|---|---|
| 0001 | pending | pending | pending |
| 0989 | pending | pending | pending |
| 1941 | pending | pending | pending |
| 2720 | pending | pending | pending |

# Step 9 — conceptual review before Git
- **What is reused?** Preprocessing → embeddings → persistent vector index → cosine retrieval → deduplication → evidence display.
- **What changed from images?** CLIP is replaced by PANNs; waveforms and resampling replace pixels and resizing.
- **What is the query?** An example audio recording, optionally constrained by known metadata. Free-form text-to-audio retrieval would need an aligned audio/text encoder such as CLAP; it is not implemented here.
- **Where is generation?** Explanations are generated by a deterministic template from retrieved scores/metadata. There is no LLM generation stage; this is the retrieval foundation of an audio RAG application.
- **What is not claimed?** Perfect similarity, personal taste prediction, commercial song identification or calibrated confidence.
- **What next?** Review listening results, then decide whether to expand the catalogue or add text-to-audio queries.

# Step 10 — Git preparation (not executed)
Commit the notebook, helper code, README, catalogue metadata, attribution and setup instructions after review. Exclude local models, datasets, uploaded personal audio, runtime/token files, package copies and databases using `.gitignore`. Do not commit credentials. We have not initialized, committed, pushed or submitted anything in this step.

## Sources
- [Ironhack lab and starter notebook](https://github.com/ironhack-labs/lab-music-recommendations)
- [PANNs implementation, MIT](https://github.com/qiuqiangkong/panns_inference)
- [PANNs official checkpoints](https://zenodo.org/records/3987831)
- [Tiny AAM dataset, CC BY 4.0](https://zenodo.org/records/6771120)
- [Chroma persistent client](https://docs.trychroma.com/reference/python)
